In [ ]:
# ============================================================
# Colab-Ready: Hybrid Quantum Kernel vs Classical Kernels
# (Qiskit + Scientific Comparison, MATLAB-like rigor)
# ============================================================

# -----------------------------
# 0) Install (Colab)
# -----------------------------
!pip -q install "qiskit>=1.0" qiskit-machine-learning scikit-learn numpy pandas matplotlib

# -----------------------------
# 1) Imports
# -----------------------------
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

# Qiskit ML (modern kernel API)
from qiskit.circuit.library import zz_feature_map
from qiskit.primitives import StatevectorSampler as Sampler
from qiskit_machine_learning.state_fidelities import ComputeUncompute
from qiskit_machine_learning.kernels import FidelityQuantumKernel


# -----------------------------
# 2) Utility: dataset generator
# -----------------------------
def generate_high_dim_dataset(
    n_samples=600,
    n_features=30,
    n_informative=20,
    n_redundant=5,
    class_sep=1.2,
    random_state=42
):
    # y is {0,1} for sklearn
    X, y = make_classification(
        n_samples=n_samples,
        n_features=n_features,
        n_informative=n_informative,
        n_redundant=n_redundant,
        n_classes=2,
        class_sep=class_sep,
        flip_y=0.01,
        random_state=random_state
    )
    return X.astype(np.float64), y.astype(int)


# -----------------------------
# 3) Utility: build quantum kernel
# -----------------------------
def build_quantum_kernel(n_qubits, reps=2, entanglement="linear"):
    # Feature map used in Qiskit kernel tutorial is zz_feature_map :contentReference[oaicite:1]{index=1}
    feature_map = zz_feature_map(
        feature_dimension=n_qubits,
        reps=reps,
        entanglement=entanglement
    )

    # StatevectorSampler + ComputeUncompute (default-modern workflow) :contentReference[oaicite:2]{index=2}
    sampler = Sampler()
    fidelity = ComputeUncompute(sampler=sampler)

    qkernel = FidelityQuantumKernel(
        fidelity=fidelity,
        feature_map=feature_map,
        enforce_psd=True,
        evaluate_duplicates="off_diagonal"
    )
    return qkernel


# -----------------------------
# 4) One experiment run (single seed)
# -----------------------------
def run_once(
    seed=42,
    n_samples=600,
    n_features=30,
    n_qubits=5,
    reps=2,
    class_sep=1.2,
    train_ratio=0.7
):
    # Reproducibility
    rng = np.random.default_rng(seed)

    # 4.1 Generate data
    X, y = generate_high_dim_dataset(
        n_samples=n_samples,
        n_features=n_features,
        n_informative=min(20, n_features),
        n_redundant=min(5, max(0, n_features - 20)),
        class_sep=class_sep,
        random_state=seed
    )

    # 4.2 Shuffle + split
    idx = rng.permutation(n_samples)
    X, y = X[idx], y[idx]

    n_train = int(train_ratio * n_samples)
    X_train, X_test = X[:n_train], X[n_train:]
    y_train, y_test = y[:n_train], y[n_train:]

    # 4.3 Standardize (fit on train only)
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s  = scaler.transform(X_test)

    # 4.4 PCA -> n_qubits
    pca = PCA(n_components=n_qubits, random_state=seed)
    X_train_q = pca.fit_transform(X_train_s)
    X_test_q  = pca.transform(X_test_s)

    # 4.5 Quantum kernel matrices
    qkernel = build_quantum_kernel(n_qubits=n_qubits, reps=reps, entanglement="linear")

    t0 = time.time()
    K_train = qkernel.evaluate(x_vec=X_train_q)
    K_test  = qkernel.evaluate(x_vec=X_test_q, y_vec=X_train_q)
    qkernel_time = time.time() - t0

    # 4.6 Quantum-kernel SVM (precomputed)
    t0 = time.time()
    svm_q = SVC(kernel="precomputed")
    svm_q.fit(K_train, y_train)
    y_pred_q = svm_q.predict(K_test)
    qsvm_time = time.time() - t0
    acc_q = accuracy_score(y_test, y_pred_q)

    # 4.7 Classical SVM baselines
    t0 = time.time()
    svm_rbf = SVC(kernel="rbf", gamma="scale")
    svm_rbf.fit(X_train_s, y_train)
    y_pred_rbf = svm_rbf.predict(X_test_s)
    rbf_time = time.time() - t0
    acc_rbf = accuracy_score(y_test, y_pred_rbf)

    t0 = time.time()
    svm_poly = SVC(kernel="poly", degree=3, gamma="scale")
    svm_poly.fit(X_train_s, y_train)
    y_pred_poly = svm_poly.predict(X_test_s)
    poly_time = time.time() - t0
    acc_poly = accuracy_score(y_test, y_pred_poly)

    return {
        "seed": seed,
        "acc_quantum_kernel": acc_q,
        "acc_rbf": acc_rbf,
        "acc_poly": acc_poly,
        "time_qkernel_eval_s": qkernel_time,
        "time_qsvm_fit_predict_s": qsvm_time,
        "time_rbf_fit_predict_s": rbf_time,
        "time_poly_fit_predict_s": poly_time,
        "y_test": y_test,
        "y_pred_q": y_pred_q,
        "y_pred_rbf": y_pred_rbf,
        "y_pred_poly": y_pred_poly,
    }


# -----------------------------
# 5) Run multiple trials (paper-style mean ± std)
# -----------------------------
N_TRIALS = 5

CONFIG = dict(
    n_samples=500,      # keep moderate for Colab speed
    n_features=30,      # high-dimensional
    n_qubits=5,         # state dimension = 2^5 = 32 (safe)
    reps=2,
    class_sep=1.2,
    train_ratio=0.7
)

results = []
last_run = None

for i in range(N_TRIALS):
    seed = 100 + i
    out = run_once(seed=seed, **CONFIG)
    last_run = out
    results.append({
        "seed": out["seed"],
        "acc_quantum_kernel": out["acc_quantum_kernel"],
        "acc_rbf": out["acc_rbf"],
        "acc_poly": out["acc_poly"],
        "time_qkernel_eval_s": out["time_qkernel_eval_s"],
        "time_qsvm_fit_predict_s": out["time_qsvm_fit_predict_s"],
        "time_rbf_fit_predict_s": out["time_rbf_fit_predict_s"],
        "time_poly_fit_predict_s": out["time_poly_fit_predict_s"],
    })
    print(f"[Trial {i+1}/{N_TRIALS}] acc(QK)={out['acc_quantum_kernel']:.3f} | acc(RBF)={out['acc_rbf']:.3f} | acc(POLY)={out['acc_poly']:.3f}")

df = pd.DataFrame(results)

print("\n==================== SUMMARY (mean ± std) ====================")
for col in ["acc_quantum_kernel", "acc_rbf", "acc_poly"]:
    print(f"{col:>20s}: {df[col].mean():.4f} ± {df[col].std(ddof=1):.4f}")

print("\n==================== RUNTIME (mean seconds) ===================")
for col in ["time_qkernel_eval_s","time_qsvm_fit_predict_s","time_rbf_fit_predict_s","time_poly_fit_predict_s"]:
    print(f"{col:>20s}: {df[col].mean():.4f}")

df


# -----------------------------
# 6) Confusion Matrices (from last trial)
# -----------------------------
y_test = last_run["y_test"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

cm_q = confusion_matrix(y_test, last_run["y_pred_q"])
ConfusionMatrixDisplay(cm_q).plot(ax=axes[0], colorbar=False)
axes[0].set_title("Quantum Kernel SVM")

cm_rbf = confusion_matrix(y_test, last_run["y_pred_rbf"])
ConfusionMatrixDisplay(cm_rbf).plot(ax=axes[1], colorbar=False)
axes[1].set_title("RBF SVM")

cm_poly = confusion_matrix(y_test, last_run["y_pred_poly"])
ConfusionMatrixDisplay(cm_poly).plot(ax=axes[2], colorbar=False)
axes[2].set_title("Polynomial SVM")

plt.tight_layout()
plt.show()
